# Навчання та оцінка обох трансформерних моделей виявлення дезінформації (`youscan/ukr-roberta-base` + `xlm-roberta-base`)

**Проєкт:** Інтелектуальний агрегатор новин з функцією виявлення неправдивої інформації (*TruthLens*)  
**Етап:** Тиждень 3 — Навчання, оцінка якості та експорт обох трансформерів (Google Colab GPU T4)  

Цей ноутбук реалізує повний цикл файн-тюнінгу **двох трансформерів одночасно**:
1. **`youscan/ukr-roberta-base`** — спеціалізована українськомовна модель для глибокого морфолого-синтаксичного аналізу українських новин.
2. **`xlm-roberta-base`** — мультимовна (cross-lingual) модель для аналізу англомовних, транскордонних та змішаних інфоводів.

Обидві моделі навчаються на збалансованому датасеті (`fake_news_dataset.csv`), оцінюються за метриками **Accuracy, Precision, Recall, F1-Score, ROC-AUC** та експортуються в окремі архіви для інтеграції у FastAPI-сервіс `MLClassifierService`.

## Крок 1. Встановлення необхідних бібліотек
У середовищі Google Colab оберіть **Runtime → Change runtime type → T4 GPU**, після чого запустіть комірку нижче.

In [ ]:
!pip install -q --upgrade transformers datasets accelerate evaluate scikit-learn matplotlib

## Крок 2. Завантаження датасету (`fake_news_dataset.csv`)
Завантажте файл `backend/ml/data/fake_news_dataset.csv` зі свого комп'ютера у вікні вибору файлу.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

DATASET_FILE = 'fake_news_dataset.csv'
if not os.path.exists(DATASET_FILE):
    try:
        from google.colab import files
        print('Завантажте файл fake_news_dataset.csv із папки backend/ml/data/:')
        uploaded = files.upload()
        DATASET_FILE = list(uploaded.keys())[0]
    except ImportError:
        DATASET_FILE = '../backend/ml/data/fake_news_dataset.csv'

df = pd.read_csv(DATASET_FILE)
print(f'Загальна кількість записів: {len(df)}')
print(df['label_name'].value_counts())

train_df, test_df = train_test_split(
    df[['combined_text', 'label']],
    test_size=0.22,
    random_state=42,
    stratify=df['label']
)
print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')
df.head(5)

## Крок 3. Універсальна функція файн-тюнінгу та оцінки трансформера

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

id2label = {0: 'Real', 1: 'Fake'}
label2id = {'Real': 0, 'Fake': 1}
MAX_LEN = 256

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs)
    return {
        'accuracy': round(float(acc), 4),
        'precision': round(float(precision), 4),
        'recall': round(float(recall), 4),
        'f1': round(float(f1), 4),
        'roc_auc': round(float(auc), 4)
    }

def train_and_export_transformer(model_hub_name: str, export_dir: str, epochs: int = 5):
    print(f'\n======================================================')
    print(f'Початок навчання моделі: {model_hub_name}')
    print(f'======================================================')
    tokenizer = AutoTokenizer.from_pretrained(model_hub_name)

    def tokenize_fn(batch):
        return tokenizer(
            batch['combined_text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN
        )

    train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize_fn, batched=True)
    test_ds = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize_fn, batched=True)
    train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

    model = AutoModelForSequenceClassification.from_pretrained(
        model_hub_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f'./results_{export_dir}',
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        fp16=torch.cuda.is_available(),
        logging_steps=10,
        report_to='none'
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_metrics = trainer.evaluate()
    print(f'=== Фінальні метрики {model_hub_name} ===')
    print(json.dumps(eval_metrics, indent=2))

    save_path = f'./{export_dir}'
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f'Модель та токенізатор збережено у {save_path}')
    return eval_metrics

## Крок 4. Навчання та експорт моделі №1: `youscan/ukr-roberta-base`
Після завершення розпакуйте архів `ukr_roberta_export.zip` у папку `backend/ml/models/ukr_roberta/`.

In [ ]:
metrics_ukr = train_and_export_transformer('youscan/ukr-roberta-base', 'ukr_roberta_export', epochs=5)
!zip -r ukr_roberta_export.zip ./ukr_roberta_export

## Крок 5. Навчання та експорт моделі №2: `xlm-roberta-base` (Мультимовний трансформер)
Після завершення розпакуйте архів `xlm_roberta_export.zip` у папку `backend/ml/models/xlm_roberta/`.

In [ ]:
metrics_xlm = train_and_export_transformer('xlm-roberta-base', 'xlm_roberta_export', epochs=5)
!zip -r xlm_roberta_export.zip ./xlm_roberta_export

## Крок 6. Порівняльна таблиця обох трансформерів (`ukr-roberta-base` vs `xlm-roberta-base`)

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': 'youscan/ukr-roberta-base',
        'Accuracy': metrics_ukr.get('eval_accuracy'),
        'Precision': metrics_ukr.get('eval_precision'),
        'Recall': metrics_ukr.get('eval_recall'),
        'F1-Score': metrics_ukr.get('eval_f1'),
        'ROC-AUC': metrics_ukr.get('eval_roc_auc'),
        'Eval Loss': metrics_ukr.get('eval_loss'),
    },
    {
        'Model': 'xlm-roberta-base',
        'Accuracy': metrics_xlm.get('eval_accuracy'),
        'Precision': metrics_xlm.get('eval_precision'),
        'Recall': metrics_xlm.get('eval_recall'),
        'F1-Score': metrics_xlm.get('eval_f1'),
        'ROC-AUC': metrics_xlm.get('eval_roc_auc'),
        'Eval Loss': metrics_xlm.get('eval_loss'),
    }
])
comparison_df